In [1]:
from google.cloud import bigquery

PROJECT = "gridzero-489711"
DATASET = "merged_set"
TABLE = "full_feature_engineered_data_test"

query = f"""
    SELECT *
    FROM {PROJECT}.{DATASET}.{TABLE}
"""

client = bigquery.Client('gridzero-489711')
query_job = client.query(query)
result = query_job.result()
df = result.to_dataframe()

/Users/jamesla/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [2]:
df.head()

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,precipitation_mm,...,carbon_intensity_gco2_kwh,hour_sin,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos,carbon_lag_48,carbon_lag_336,carbon_lag_17520
0,2017-09-12 00:00:00+00:00,11.6,31.0,28.1,4.0,0.0,0.0,0.0,1001.2,0.0,...,142.0,0.000000,1.000000,0.781831,0.62349,-0.948362,-0.317191,NaN,NaN,NaN
1,2017-09-12 00:30:00+00:00,11.6,31.0,28.1,4.0,0.0,0.0,0.0,1001.2,0.0,...,140.0,0.000000,1.000000,0.781831,0.62349,-0.948362,-0.317191,NaN,NaN,NaN
2,2017-09-12 01:00:00+00:00,11.2,30.3,27.0,5.0,0.0,0.0,0.0,1001.9,0.0,...,139.0,0.258819,0.965926,0.781831,0.62349,-0.948362,-0.317191,NaN,NaN,NaN
3,2017-09-12 01:30:00+00:00,11.2,30.3,27.0,5.0,0.0,0.0,0.0,1001.9,0.0,...,137.0,0.258819,0.965926,0.781831,0.62349,-0.948362,-0.317191,NaN,NaN,NaN
4,2017-09-12 02:00:00+00:00,10.9,29.6,25.2,7.0,0.0,0.0,0.0,1002.4,0.0,...,132.0,0.500000,0.866025,0.781831,0.62349,-0.948362,-0.317191,NaN,NaN,NaN


In [3]:
df.columns

Index(['datetime', 'temperature_2m_c', 'wind_speed_100m_ms',
       'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2',
       'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa',
       'precipitation_mm', 'biomass', 'fossil_gas', 'fossil_hard_coal',
       'hydro_pumped_storage', 'hydro_run_of_river_and_poundage', 'nuclear',
       'other', 'solar', 'wind_offshore', 'wind_onshore', 'totaloutput_mw',
       'carbon_intensity_gco2_kwh', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'doy_sin', 'doy_cos', 'carbon_lag_48', 'carbon_lag_336',
       'carbon_lag_17520'],
      dtype='object')

In [4]:
# feature_cols = [

# # weather
# 'temperature_2m_c',
# 'wind_speed_100m_ms',
# 'cloud_cover_pct',
# 'shortwave_radiation_wm2',

# # time
# 'hour_sin','hour_cos',
# 'dow_sin','dow_cos',
# 'doy_sin','doy_cos',

# # lagged generation
# 'solar_lag_48',
# 'wind_onshore_lag_48',
# 'wind_offshore_lag_48'
# ]

In [5]:
feature_cols = [
    # weather
    'temperature_2m_c',
    'wind_speed_100m_ms',
    'wind_gusts_10m_ms',
    'cloud_cover_pct',
    'shortwave_radiation_wm2',
    'direct_radiation_wm2',
    'diffuse_radiation_wm2',
    'pressure_msl_hpa',
    'precipitation_mm',

    # time
    'hour_sin','hour_cos',
    'dow_sin','dow_cos',
    'doy_sin','doy_cos',
    
    # past generation (important)
    'biomass',
    'fossil_gas',
    'fossil_hard_coal',
    'hydro_pumped_storage',
    'hydro_run_of_river_and_poundage',
    'nuclear',
    'other',
    'solar',
    'wind_offshore',
    'wind_onshore'
]

In [6]:
target_cols = [
    'biomass',
    'fossil_gas',
    'fossil_hard_coal',
    'hydro_pumped_storage',
    'hydro_run_of_river_and_poundage',
    'nuclear',
    'other',
    'solar',
    'wind_offshore',
    'wind_onshore'
]

In [7]:
from sklearn.preprocessing import MinMaxScaler

X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X = X_scaler.fit_transform(df[feature_cols])
y = y_scaler.fit_transform(df[target_cols])

In [8]:
import numpy as np

lookback = 336

def create_sequences(X, y, lookback):
    
    Xs, ys = [], []
    
    for i in range(len(X) - lookback):
        Xs.append(X[i:i+lookback])
        ys.append(y[i+lookback])
        
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X, y, lookback)

In [9]:
X_seq.shape, y_seq.shape

((148655, 336, 25), (148655, 10))

In [10]:
train_size = int(len(X_seq) * 0.7)
val_size = int(len(X_seq) * 0.15)

X_train = X_seq[:train_size]
y_train = y_seq[:train_size]

X_val = X_seq[train_size:train_size+val_size]
y_val = y_seq[train_size:train_size+val_size]

X_test = X_seq[train_size+val_size:]
y_test = y_seq[train_size+val_size:]

## Building the LSTM model

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(LSTM(128, return_sequences=True,
               input_shape=(lookback, len(feature_cols))))

model.add(Dropout(0.2))

model.add(LSTM(64))

model.add(Dense(64, activation="relu"))

model.add(Dense(len(target_cols)))

In [16]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="mse",
    metrics=["mae"]
)

In [ ]:
# model.compile(
#     optimizer=Adam(learning_rate=0.001),
#     loss="mse"
# )

In [14]:
# history = model.fit(
#     X_train,
#     y_train,
#     validation_data=(X_val, y_val),
#     epochs=40,
#     batch_size=64
# )

In [17]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath="checkpoints/best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

In [18]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [ ]:
model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[checkpoint, early_stop]
)

In [ ]:
model.save("lstm_models/model1")

In [ ]:
model = load_model("lstm_models/model2")